In [2]:
import os

import pandas as pd
import numpy as np

In [86]:
SUBSET = 'others'  # "eu" or "others"
RESULTS_DIR = '../experiments_ancestry'
EU_ID = 6
MIN_NUMBER_CASES_PER_SYNDROME = 1

In [87]:
if SUBSET == 'eu':
    print("Gathering results for EU+EU* models")
elif SUBSET == 'others':
    print("Gathering results EU+Others models")
else:
    print(f"Unknown set given (got subset '{SUBSET}')")
    exit()

ancs = []
tars = []
preds = []
for i in range(1,6):
    if SUBSET == 'eu':
        file_path = os.path.join(RESULTS_DIR, f'results_s811{10+i}_seed{i}_eu.npy')
    elif SUBSET == 'others':
        file_path = os.path.join(RESULTS_DIR, f'results_s8110{i}_seed{i}_all_eth.npy')

    # [idx, ethn_id, target_disorder, pred_disorders]
    _, anc, tar, pred = np.load(file_path, allow_pickle=True)

    ancs.append(np.hstack(anc).astype(int))
    tars.append(np.hstack(tar).astype(int))
    preds.append(pred)

df = pd.DataFrame({'ancestry': ancs[-1], 'syndrome': tars[-1]})

print(f"Syndrome ID value counts:")
print(df.syndrome.value_counts())

Gathering results EU+Others models
Syndrome ID value counts:
0      38
1      24
35     24
3      19
11     19
       ..
175     1
120     1
104     1
132     1
98      1
Name: syndrome, Length: 175, dtype: int64


In [88]:
# Create df with number of cases per synd for each ancestry
counts = df.groupby(['syndrome', 'ancestry']).size().reset_index(name='count')
anc_freq_df = counts.pivot(index='syndrome', columns='ancestry', values='count')
anc_freq_df = anc_freq_df.fillna(0).astype(int)
anc_freq_df = anc_freq_df.reset_index()

local_metadata_path = 'C:/Users/alexa/Documents/Workspace/data/GestaltMatcherDB/v1.1.0/gmdb_metadata/'

# Rename syndrome IDs to syndrome names
synd_lookup = pd.read_csv(os.path.join(local_metadata_path, 'gmdb_syndromes_v1.1.0.tsv'), sep='\t', usecols=['syndrome_id', 'syndrome_name'])
anc_freq_df = anc_freq_df.merge(synd_lookup, left_on='syndrome', right_on='syndrome_id', how='left')
anc_freq_df = anc_freq_df.drop(columns=['syndrome'])  # optional cleanup
anc_freq_df = anc_freq_df.rename(columns={'syndrome_name': 'syndrome'})

# Rename ancestry IDs to ancestry names
anc_lookup = ['African American', 'African Others', 'American - Latin/Hispanic', 'American - Native', 'Asian Others', 'East Asian', 'European', 'Middle-East/West Asian', 'Mixed ancestry', 'North African', 'Others', 'South Asian', 'South-East Asian', 'Sub-Saharan']
ancestry_map = {i: name for i, name in enumerate(anc_lookup)}
anc_freq_df = anc_freq_df.rename(columns=ancestry_map)

# Add a non-European column
anc_freq_df['non-European'] = anc_freq_df.drop(columns=['syndrome', 'syndrome_id', 'European']).sum(axis=1)

# Re-order to have the most relevant columns first
cols = ['syndrome', 'syndrome_id', 'European', 'non-European'] + [c for c in anc_freq_df.columns if c not in ['syndrome', 'syndrome_id', 'European', 'non-European']]
anc_freq_df = anc_freq_df[cols]

anc_freq_df[(anc_freq_df['European'] >= MIN_NUMBER_CASES_PER_SYNDROME) & (anc_freq_df['non-European'] >= MIN_NUMBER_CASES_PER_SYNDROME)][['syndrome', 'syndrome_id', 'European', 'non-European']]


,syndrome,syndrome_id,European,non-European
0,Cornelia de Lange syndrome,0,23,15
1,WILLIAMS-BEUREN SYNDROME; WBS,1,4,20
2,Noonan syndrome,2,9,3
3,Kabuki syndrome,3,12,7
4,Coffin-Siris syndrome,4,10,2
5,KBG SYNDROME; KBGS,5,15,2
6,ANGELMAN SYNDROME; AS,6,13,5
7,WIEDEMANN-STEINER SYNDROME; WDSTS,7,6,4
8,Mucopolysaccharidoses,8,7,2
9,Rubinstein-Taybi syndrome,9,7,4


In [89]:
# European ancestry has id 6
eu_mask = (df.ancestry == EU_ID).values
eu_idxs = df[df.ancestry == EU_ID].index.values
non_eu_mask = (df.ancestry != EU_ID).values
non_eu_idxs = df[df.ancestry != EU_ID].index.values

eu_synds = np.array(df[df.ancestry == EU_ID].syndrome.value_counts().keys())
sort_idxs = eu_synds.argsort()
eu_synds = eu_synds[sort_idxs]
eu_synds_count = df[df.ancestry == EU_ID].syndrome.value_counts().values[sort_idxs]

non_eu_synds = np.array(df[df.ancestry != EU_ID].syndrome.value_counts().keys())
sort_idxs = non_eu_synds.argsort()
non_eu_synds = non_eu_synds[sort_idxs]
non_eu_synds_count = df[df.ancestry != EU_ID].syndrome.value_counts().values[sort_idxs]

print(f"Syndrome ids in both EU and Other subsets: {eu_synds[np.isin(eu_synds, non_eu_synds)]}")
print(f"Frequency of test images for patients with EU ancestry: {eu_synds_count[np.isin(eu_synds, non_eu_synds)]}")
print(f"Frequency of test images for patients with Other ancestries: {non_eu_synds_count[np.isin(non_eu_synds, eu_synds)]}")

Syndrome ids in both EU and Other subsets: [  0   1   2   3   4   5   6   7   8   9  10  11  13  14  15  16  17  18
  19  21  22  26  27  28  30  31  32  35  36  37  39  40  42  46  49  50
  53  55  56  57  58  64  65  66  69  70  73  75  78  79  92  96 100 101
 145 152 164]
Frequency of test images for patients with EU ancestry: [23  4  9 12 10 15 13  6  7  7  6 12  6  2  1  1  3 11  7  1  3  1  4  2
  2  2  1 22  5  5  1  3  2  2  3  1  4  2  3  2  2  1  1  1  1  5  1  1
  1  1  2  1  2  1  1  1  1]
Frequency of test images for patients with Other ancestries: [15 20  3  7  2  2  5  4  2  4  4  7  1  5  5  2  2  1  1  6  4  3  1  1
  1  1  1  2  1  1  1  2  1  2  1  3  1  2  3  1  1  2  3  2  2  1  3  1
  1  1  3  1  1  2  1  2  1]


In [90]:
overlapping_synd_ids = eu_synds[np.isin(eu_synds, non_eu_synds)]
test_freq_df = pd.DataFrame({'synd_id': overlapping_synd_ids,
                    'eu_freq': eu_synds_count[np.isin(eu_synds, non_eu_synds)],
                    'non_eu_freq': non_eu_synds_count[np.isin(non_eu_synds, eu_synds)]
                    })

# test_performance_df = ...
ranks = np.array([np.where(t[0] == p)[0][0] for p,t in zip(pred, tar)])
# Example top-N for N=10
# len(ranks[ranks<10]) / len(ranks)

# Top-N accuracy of EU-only and Other-only; averaged over all images - some EU/Other synds are exclusive
print("\nTop-N accuracy averaged over all images, for all synds")
for idx, subset in enumerate([eu_synds, non_eu_synds]):
    subset_idxs = eu_idxs if idx == 0 else non_eu_idxs
    print(f"{'EU' if idx == 0 else 'Other'}-only:")
    for n in [1,5,10]:
        print(f"\tTop-{n}: {sum(ranks[subset_idxs] < n) / len(subset_idxs)}")



Top-N accuracy averaged over all images, for all synds
EU-only:
	Top-1: 0.5305164319248826
	Top-5: 0.715962441314554
	Top-10: 0.7887323943661971
Other-only:
	Top-1: 0.6633663366336634
	Top-5: 0.801980198019802
	Top-10: 0.8564356435643564


In [91]:
# Top-N accuracy of EU-only and Other-only; averaged over all images - only synds of both EU and Other
print("\nTop-N accuracy averaged over all images, for common synds")
for idx, subset in enumerate([eu_synds, non_eu_synds]):
    subset_mask = eu_mask if idx == 0 else non_eu_mask
    common_synd_ranks = ranks[np.isin(tar, overlapping_synd_ids) & subset_mask]
    print(f"{'EU' if idx == 0 else 'Other'}-only (num_imgs={len(common_synd_ranks)}):")
    for n in [1,5,10]:
        print(f"\tTop-{n}: {sum(common_synd_ranks < n) / len(common_synd_ranks)}")


Top-N accuracy averaged over all images, for common synds
EU-only (num_imgs=250):
	Top-1: 0.624
	Top-5: 0.78
	Top-10: 0.828
Other-only (num_imgs=159):
	Top-1: 0.7358490566037735
	Top-5: 0.8490566037735849
	Top-10: 0.8930817610062893


In [92]:
n = 5
# overlapping_synd_ids_frequent = [0,1,2,3,6,7,9,10,11,22]
# overlapping_synd_ids_frequent = [0,6,3,11]
overlapping_synd_ids = anc_freq_df[(anc_freq_df['European'] >= MIN_NUMBER_CASES_PER_SYNDROME) & (anc_freq_df['non-European'] >= MIN_NUMBER_CASES_PER_SYNDROME)][['syndrome', 'syndrome_id', 'European', 'non-European']].syndrome_id.values

# Top-N accuracy per synd of EU-only and Other-only; averaged over all images - only synds of both EU and Other
print(f"\nTop-{n} accuracy per synd, for common synds")
accs = []
for idx, subset in enumerate([eu_synds, non_eu_synds]):
    accs_all_synds = []
    subset_mask = eu_mask if idx == 0 else non_eu_mask
    print(f"\n{'EU' if idx == 0 else 'Other'}-only:")
    for synd_id in overlapping_synd_ids:#[:5]:
    # for synd_id in overlapping_synd_ids_frequent:
        synd_idxs = (df.syndrome == synd_id).values
        synd_ranks = ranks[subset_mask & synd_idxs]
        if synd_id < 500:
            print(f"{(sum(synd_ranks < n) / len(synd_ranks))*100:.2f}%\t({(anc_freq_df[anc_freq_df.syndrome_id == synd_id]['European' if idx == 0 else 'non-European'].values[0]):2})\t{anc_freq_df[anc_freq_df.syndrome_id == synd_id].syndrome.values[0]}")
        accs_all_synds.append(sum(synd_ranks < n) / len(synd_ranks))
    accs.append(accs_all_synds)
print(f"Mean top-{n} accuracy over all overlapping syndromes of EU-only vs Other-only test images: {np.mean(accs, axis=1)}")


Top-5 accuracy per synd, for common synds

EU-only:
95.65%	(23)	Cornelia de Lange syndrome
100.00%	( 4)	WILLIAMS-BEUREN SYNDROME; WBS
77.78%	( 9)	Noonan syndrome
91.67%	(12)	Kabuki syndrome
80.00%	(10)	Coffin-Siris syndrome
73.33%	(15)	KBG SYNDROME; KBGS
100.00%	(13)	ANGELMAN SYNDROME; AS
83.33%	( 6)	WIEDEMANN-STEINER SYNDROME; WDSTS
57.14%	( 7)	Mucopolysaccharidoses
100.00%	( 7)	Rubinstein-Taybi syndrome
83.33%	( 6)	SOTOS SYNDROME; SOTOS
91.67%	(12)	OGDEN SYNDROME; OGDNS
33.33%	( 6)	Arthrogryposis, distal
0.00%	( 2)	Hyperphosphatasia with mental retardation syndrome
100.00%	( 1)	Alagille syndrome
100.00%	( 1)	PITT-HOPKINS SYNDROME; PTHS
100.00%	( 3)	HELSMOORTEL-VAN DER AA SYNDROME; HVDAS
81.82%	(11)	Hutchinson-Gilford progeria
100.00%	( 7)	FLOATING-HARBOR SYNDROME; FLHS
100.00%	( 1)	Treacher Collins syndrome
0.00%	( 3)	Cutis laxa
100.00%	( 1)	Trichorhinophalangeal syndrome
25.00%	( 4)	INTELLECTUAL DEVELOPMENTAL DISORDER, AUTOSOMAL DOMINANT 5; MRD5
50.00%	( 2)	Holoprosencephaly
50.00%

In [93]:
Ns = [1,5]
# overlapping_synd_ids_frequent = [0,1,2,3,6,7,9,10,11,22]

# Top-N accuracy per synd of EU-only and Other-only; averaged over all images - only synds of both EU and Other
print(f"Results for {'[EU + EU*]' if SUBSET=='eu' else '[EU + non-EU]'} with {MIN_NUMBER_CASES_PER_SYNDROME=}")
for n in Ns:
    print(f"\nTop-{n} accuracy per synd, for common synds")
    accs = []
    for idx, subset in enumerate([eu_synds, non_eu_synds]):
        accs_all_synds = []
        subset_mask = eu_mask if idx == 0 else non_eu_mask
        # for synd_id in overlapping_synd_ids:
        for synd_id in overlapping_synd_ids:
            synd_idxs = (df.syndrome == synd_id).values
            synd_ranks = ranks[subset_mask & synd_idxs]
            accs_all_synds.append(sum(synd_ranks < n) / len(synd_ranks))
        accs.append(accs_all_synds)
    print(f"Mean top-{n} accuracy over all overlapping syndromes of EU-only vs Other-only test images: {np.mean(accs, axis=1)}")

Results for [EU + non-EU] with MIN_NUMBER_CASES_PER_SYNDROME=1

Top-1 accuracy per synd, for common synds
Mean top-1 accuracy over all overlapping syndromes of EU-only vs Other-only test images: [0.4832978 0.6154553]

Top-5 accuracy per synd, for common synds
Mean top-5 accuracy over all overlapping syndromes of EU-only vs Other-only test images: [0.6843376  0.76269841]


In [94]:
# Average top-N EU+EU* vs EU+Others - overlapping (MIN_NUMBER_CASES_PER_SYNDROME>=1)
print("Mean performance delta of [EU + EU*] vs [EU + non-EU] for MIN_NUMBER_CASES_PER_SYNDROME=1:" )
print(f"Top-1: {(np.mean([0.49282883, 0.38721805]) - np.mean([0.4832978 , 0.6154553 ]))*100:.2f}%") #top-1
print(f"Top-5: {(np.mean([0.75737177, 0.6625731 ]) - np.mean([0.6843376 , 0.76269841]))*100:.2f}%") #top-5
print()

# Average top-N EU+EU* vs EU+Others - overlapping (MIN_NUMBER_CASES_PER_SYNDROME>=3)
print("Mean performance delta of [EU + EU*] vs [EU + non-EU] for MIN_NUMBER_CASES_PER_SYNDROME=3:" )
print(f"Top-1: {(np.mean([0.71093158, 0.55497835]) - np.mean([0.70630939, 0.77705628]))*100:.2f}%") #top-1
print(f"Top-5: {(np.mean([0.88442113, 0.76969697]) - np.mean([0.83948177, 0.88852814]))*100:.2f}%") #top-5

Mean performance delta of [EU + EU*] vs [EU + non-EU] for MIN_NUMBER_CASES_PER_SYNDROME=1:
Top-1: -10.94%
Top-5: -1.35%

Mean performance delta of [EU + EU*] vs [EU + non-EU] for MIN_NUMBER_CASES_PER_SYNDROME=3:
Top-1: -10.87%
Top-5: -3.69%
